In [ ]:
import numpy as np
import ants
import matplotlib.pyplot as plt
import os
from antspynet.utilities import brain_extraction
import logging
import gc
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import ImageGrid

2025-10-16 16:24:42.458434: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-16 16:24:42.489057: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-16 16:24:42.630182: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-16 16:24:42.785876: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-16 16:24:42.887812: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [ ]:
# Configuração do logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

def porcentagem_limiar(values, thresh): #vê qual o porcentagem de valores acima de um threshold
    if len(values.shape) > 1:
        values = values.flatten()

    thresh_count = 0
    
    for value in values:
        if value > thresh:
            thresh_count += 1

    print(f"TOTAL VOXELS: {len(values)} \nMAIOR QUE {thresh}: {thresh_count} \nrazão: {(thresh_count/len(values))*100:.2f}%")

def metricas_imagem(data): #print metricas de uma imagem (max, min, media)
    if len(data.shape) > 1:
        values = data.flatten()

    print(f"MEDIA: {np.mean(values)}")
    print(f"MIN: {np.min(values)}")
    print(f"MAX: {np.max(values)}")

    plt.hist(values)
    plt.show()

def winsorize_image(image_data, lower_percentile=0, upper_percentile=99.9): #reduz valores extremos
    lower_bound = np.percentile(image_data, lower_percentile)
    upper_bound = np.percentile(image_data, upper_percentile)
    winsorized_data = np.clip(image_data, lower_bound, upper_bound)
    return winsorized_data

def normalize_image_min(image_data): #normalizar 
    min_val = np.min(image_data)
    max_val = np.max(image_data)
    normalized_data = (image_data - min_val) / (max_val - min_val)
    return normalized_data

# Função para processar uma única imagem
def process_image(img_path, template, registro='Affine', orientation='false'):
    try:
        logger.info(f"Inicio processamento: {img_path}")
        # Carrega a imagem
        image = ants.image_read(img_path,reorient=orientation)

        # Registra pra padronizar shape da imagem
        registration = ants.registration(fixed=template, moving=image, type_of_transform=registro)
        affine_image = registration['warpedmovout']
        brain_masked = affine_image

        # Cria template pra máscara
        prob_mask = brain_extraction(affine_image, modality='t1')
        # logger.info(f"Template obtido.")

        # # Cria a máscara
        mask = ants.get_mask(prob_mask, low_thresh=0.5)
        # logger.info(f"Máscara aplicada.")

        # # Máscara do cérebro e extração
        brain_masked = ants.mask_image(affine_image, mask)
        # logger.info(f"Extração.")

        # Bias Field Correction
        #image = ants.from_numpy(data, origin=image.origin, spacing=image.spacing, direction=image.direction)
        image = ants.n4_bias_field_correction(brain_masked, shrink_factor=2)
        data = image.numpy()
        logger.info(f"Bias Corrigido.")

        # Winsorizing
        data = winsorize_image(data, 0, 99.9)
        logger.info(f"Winsorized.")

        # Normalização
        data = normalize_image_min(data)
        image = ants.from_numpy(data, origin=brain_masked.origin, spacing=brain_masked.spacing, direction=brain_masked.direction)

        logger.info(f"Imagem {img_path} processada.")

        #ants.image_write(image, output_path)
        #logger.info(f"Imagem salva: {os.path.basename(output_path)}")

        gc.collect()

        return image
        
    except Exception as e:
        logger.error(f"Erro ao processar a imagem {img_path}: {e}")
        return None
    
def plot_views(image, k=0):
    fig, axs = plt.subplots(1, 3)
    size = image.shape

    axs[0].imshow(np.rot90(image[size[0]//2, :, :], k=k), cmap='grey')
    axs[0].set_title("esperado: sagital")
    axs[0].axis('off')
    
    axs[1].imshow(np.rot90(image[:, size[1]//2, :]), cmap='grey')
    axs[1].set_title("esperado: coronal")
    axs[1].axis('off')

    axs[2].imshow(np.rot90(image[:, :, size[2]//2], k=k), cmap='grey')
    axs[2].set_title("esperado: axial")
    axs[2].axis('off')
    
    fig.tight_layout(rect=[0, 0, 1, 0.8])
    plt.show()

def plot_one_view(image, index=80, rot=0, title=''):
    image = np.rot90(image[:, :, index], k=rot)
    plt.figure(figsize=(10,6))
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

def plot_views_com_grid(image, k=0, sag_idx=90, cor_idx=110, ax_idx=110, figsize=(15, 5)):
    fig = plt.figure(figsize=figsize)

    grid = ImageGrid(fig, 111,
                    nrows_ncols=(1, 3),
                    axes_pad=0.5,
                    )
    
    slices = [
        np.rot90(image[sag_idx, :, :], k=k),
        np.rot90(image[:, cor_idx, :], k=k),
        np.rot90(image[:, :, ax_idx], k=k)
    ]
    titles = ["esperado: sagital", "esperado: coronal", "esperado: axial"]

    for ax, im_slice, title in zip(grid, slices, titles):
        ax.imshow(im_slice, cmap='gray')
        ax.set_title(title)
        ax.axis('off')

    plt.show()

def unificar_tamanhos_com_padding(lista_de_imagens):
    max_altura = 0
    max_largura = 0
    for img in lista_de_imagens:
        altura, largura = img.shape
        if altura > max_altura:
            max_altura = altura
        if largura > max_largura:
            max_largura = largura

    imagens_uniformes = []
    for img in lista_de_imagens:
        fundo = np.zeros((max_altura, max_largura))
        
        altura_img, largura_img = img.shape
        y_offset = (max_altura - altura_img) // 2
        x_offset = (max_largura - largura_img) // 2
        
        fundo[y_offset:y_offset+altura_img, x_offset:x_offset+largura_img] = img
        imagens_uniformes.append(fundo)
        
    return imagens_uniformes

def plot_views_uniforme_final(image, k=0, sag_idx=90, cor_idx=110, ax_idx=110, figsize=(15, 5), axes_pad=0.3):
    fig = plt.figure(figsize=figsize)
    grid = ImageGrid(fig, 111,
                    nrows_ncols=(1, 3),
                    axes_pad=axes_pad)
    
    slices_originais = [
        np.rot90(image[sag_idx, :, :], k=k),
        np.rot90(image[:, cor_idx, :], k=k),
        np.rot90(image[:, :, ax_idx], k=k)
    ]
    
    slices_uniformizadas = unificar_tamanhos_com_padding(slices_originais)
    
    titles = ["Sagital", "Coronal", "Axial"]

    for ax, im_slice, title in zip(grid, slices_uniformizadas, titles):
        ax.imshow(im_slice, cmap='gray')
        ax.set_title(title)
        ax.axis('off')

    plt.show()

In [ ]:
dir = '/mnt/c/Users/Bruno/Documents/Github'

# path = f"/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS-1/RAW Data/disc1/OAS1_0001_MR1/RAW/OAS1_0001_MR1_mpr-1_anon.img"
oasis_dir = "/mnt/c/Users/Bruno/Desktop/IANS/OAS2_RAW_PART1/OAS2_0001_MR1/RAW"
oasis_sample_path = f"{oasis_dir}/mpr-1.nifti.img"

adni_dir = "/mnt/c/Users/Bruno/Desktop/IANS/Alzheimer/test_affine/ad"
adni_sample_path = f"{adni_dir}/I297850.nii.gz" 

template_path = f"{dir}/Alzheimer-CNN-Detection/pre_processing/mni_icbm152_nlin_asym_09c_nifti/mni_icbm152_nlin_asym_09c/mni_icbm152_t1_tal_nlin_asym_09c.nii"
template = ants.image_read(template_path)
mask_path = f"{dir}/Alzheimer-CNN-Detection/pre_processing/mni_icbm152_nlin_asym_09c_nifti/mni_icbm152_nlin_asym_09c/mni_icbm152_t1_tal_nlin_asym_09c_mask.nii"
mask = ants.image_read(mask_path)

In [ ]:
dimension = ants.image_read(adni_sample_path)
print(dimension.shape)

(193, 229, 193)


In [ ]:
oasis_proc_dir = "/mnt/c/Users/Bruno/Desktop/IANS/OASIS_2_PROCESSED"

print(os.listdir(oasis_proc_dir))

In [ ]:
for sub in os.listdir(oasis_proc_dir):
    names = os.listdir(f"{oasis_proc_dir}/{sub}")
    for name in names:
        img = ants.image_read(f"{oasis_proc_dir}/{sub}/{name}")
        plot_views_uniforme_final(img.numpy(), 0, 86, 140, 74, (15,5), 0.3)

In [6]:
names = os.listdir(f"{adni_dir}")
for name in names:
    img = ants.image_read(f"{adni_dir}/{name}")
    plot_views_uniforme_final(img.numpy(), 0, 86, 140, 74, (15,5), 0.3)

KeyboardInterrupt: 

In [ ]:
# Carregando imagem original
adni_sample_img = ants.image_read(adni_sample_path)
plot_views_uniforme_final(adni_sample_img.numpy(), 0, 89, 94, 88, (15,5), 0.3)

oasis_sample_img = ants.image_read(oasis_sample_path)
plot_views_uniforme_final(oasis_sample_img.numpy(), 0, 89, 94, 88, (15,5), 0.3)

In [ ]:
# Registra pra padronizar shape da imagem
registration = ants.registration(fixed=template, moving=oasis_sample_img, type_of_transform='Affine')
affine_image = registration['warpedmovout']

In [ ]:
plot_views_uniforme_final(affine_image.numpy(), 0, 89, 94, 88, (15,5), 0.3)

In [ ]:
# Cria template pra máscara
prob_mask = brain_extraction(affine_image, modality='t1')

# # Cria a máscara
mask = ants.get_mask(prob_mask, low_thresh=0.5)

# # Máscara do cérebro e extração
brain_masked = ants.mask_image(affine_image, mask)

In [ ]:
plot_views_uniforme_final(brain_masked.numpy(), 0, 89, 94, 88, (15,5), 0.3)

In [ ]:
# Bias Field Correction
bias_image = ants.n4_bias_field_correction(brain_masked, shrink_factor=2)
bias_data = bias_image.numpy()

# Winsorizing
wins_data = winsorize_image(bias_data, 0, 99.9)

# Normalização
norm_data = normalize_image_min(wins_data)

In [ ]:
plot_views_uniforme_final(norm_data, 0, 89, 94, 95, (15,5), 0.3)

plot_views_uniforme_final(oasis_sample_img.numpy(), 0, 86, 140, 74, (15,5), 0.3)

In [ ]:
# plot_views_com_grid(norm_data, 1, 89, 94, 88, (15,5))
plot_views_uniforme_final(norm_data, 1, 89, 94, 88, (15,5), 0.3)

In [ ]:
# translation
translation_image = process_image(path, template, 'Translation', 'IRA')

# translation
print('TRANSLATION')
for i in range(0, translation_image.shape[2], 10):
    plt.imshow(translation_image.numpy()[:, :, i], cmap='gray')
    plt.title(translation_image.shape)
    plt.show()

In [ ]:
# rigid
rigid_image = process_image(path, template, 'Rigid')

# rigid
print('RIGID')
for i in range(0, translation_image.shape[2], 10):
    plt.imshow(rigid_image.numpy()[:, :, i], cmap='gray')
    plt.title(rigid_image.shape)
    plt.show()

In [ ]:
#affine
affine_image = process_image(path, template, 'Affine')

#affine
print('AFFINE')
for i in range(0, translation_image.shape[2], 10):
    plt.imshow(affine_image.numpy()[:, :, i], cmap='gray')
    plt.title(affine_image.shape)
    plt.show()

In [ ]:
# original
print('ORIGINAL')
for i in range(0, img_data.shape[2], 10):
    plt.imshow(img_data[:, :, i], cmap='gray')
    plt.title(img_data.shape)
    plt.show()

In [ ]:
# DADOS DA IMAGEM PURA NÃO PROCESSADA
metricas_imagem(img_data)
#porcentagem_limiar(img_data, 2300)

In [ ]:
# DADOS DA IMAGEM NORMALIZADA SEM WINSORIZATION
norm_sem_wins_data = normalize_image_min(img_data)
metricas_imagem(norm_sem_wins_data)

In [ ]:
# DADOS DA IMAGEM PURA WINSORIZADA
wins_data = winsorize_image(img_data, 0, 99.9)
metricas_imagem(wins_data)

In [ ]:
# DADOS DA IMAGEM WINSORIZADA E NORMALIZADA
norm_data = normalize_image_min(wins_data)
metricas_imagem(norm_data)

In [ ]:
dir = '/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/to_process/NIFTI_RAW/test'

count = 0

for subset in os.listdir(dir):
    print("="*300)
    print(f"{subset}")
    print("="*300)
    
    count = 0
    names_path = f"{dir}/{subset}"
    names = os.listdir(names_path)

    for name in names:
        while count < 15:
            filepath = f"{names_path}/{name}"
            img = ants.image_read(filepath)
            img_data = img.numpy()

            metricas_imagem(img_data)

            count += 1

In [ ]:
count = 0

for subset in os.listdir(dir):
    print("="*300)
    print(f"{subset}")
    print("="*300)
    
    count = 0
    names_path = f"{dir}/{subset}"
    names = os.listdir(names_path)

    for name in names:
        if count < 15:
            filepath = f"{names_path}/{name}"
            img_for = ants.image_read(filepath)
            img_data_for = img_for.numpy()
            img_data_for = winsorize_image(img_data_for, 0, 99.9)

            metricas_imagem(img_data_for)

            count += 1

In [ ]:
plt.imshow(img_data[:, :, 133], cmap="gray")
plt.title(img_data.shape)
plt.show()

In [ ]:
plt.imshow(template.numpy()[:, :, 80], cmap="gray")
plt.title(template.shape)
plt.show()

#### Testando TRANSLATION na mri pura

In [ ]:
registration = ants.registration(fixed=template, moving=img, type_of_transform='Translation')
trans_image = registration['warpedmovout']

In [ ]:
plt.imshow(trans_image.numpy()[:, :, 80], cmap="gray")
plt.title(trans_image.shape)
plt.show()

#### Testando RIGID na mri pura

In [ ]:
registration = ants.registration(fixed=template, moving=img, type_of_transform='Rigid')
rigid_image = registration['warpedmovout']

In [ ]:
plt.imshow(rigid_image.numpy()[:, :, 80], cmap="gray")
plt.title(rigid_image.shape)
plt.show()

#### Testando AFFINE na mri pura

In [ ]:
registration = ants.registration(fixed=template, moving=img, type_of_transform='Affine')
affine_image = registration['warpedmovout']

In [ ]:
plt.imshow(affine_image.numpy()[:, :, 80], cmap="gray")
plt.imshow(mask.numpy()[:, :, 80], cmap='gray', alpha=0.0)
plt.title(affine_image.shape)
plt.show()

In [ ]:
prob_mask = brain_extraction(affine_image, modality='t1')

# Cria a máscara
mask = ants.get_mask(prob_mask, low_thresh=0.5)

# Máscara do cérebro e extração
brain_masked = ants.mask_image(affine_image, mask)

#### Testando SyN na mri pura

In [ ]:
registration = ants.registration(fixed=template, moving=img, type_of_transform='SyN')
syn_image = registration['warpedmovout']

In [ ]:
plt.imshow(syn_image.numpy()[:, :, 80], cmap="gray")
plt.imshow(mask.numpy()[:, :, 80], cmap='gray', alpha=0.25)
plt.title(syn_image.shape)
plt.show()

In [ ]:
affine_data = affine_image.numpy()
metricas_imagem(affine_data)

In [ ]:
affine_norm = winsorize_image(affine_data, 1, 99.9)
metricas_imagem(affine_norm)

In [ ]:
plt.imshow(affine_norm[:, :, 80], cmap='gray')
plt.show()

In [ ]:
dir = 'test_raw'

folders = os.listdir(dir)

for subset in folders:
    count = 0
    subset_path = f"{dir}/{subset}"

    names = os.listdir(subset_path)
    for name in names:
        while count < 1:
            name_path = f"{subset_path}/{name}"
            pre_data = ants.image_read(name_path)

            registration = ants.registration(fixed=template, moving=pre_data, type_of_transform='Rigid')
            rigid_data = registration['warpedmovout']

            # Cria template pra máscara
            prob_mask = brain_extraction(affine_data, modality='t1') # MODALIDADE: 'flair' ou 't1'

            # Cria a máscara
            mask = ants.get_mask(prob_mask, low_thresh=0.5)

            # Máscara do cérebro e extração
            brain_masked = ants.mask_image(affine_data, mask)

            wins_data = winsorize_image(affine_data.numpy(), 1, 99.9)

            norm_wins_data = normalize_image_min(wins_data)

            plt.imshow(norm_wins_data[:, :, 80], cmap='gray')
            plt.title(norm_wins_data.shape)
            plt.show()

            metricas_imagem(norm_wins_data)
            count += 1